[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/nn_diabetes.ipynb)

# Prediction of Diabetes using ANN

The dataset is from the National Institute of Diabetes and Digestive and Kidney Diseases. The objective is to predict whether a patient has diabetes based on diagnostic measurements. All patients are females of Pima Indian heritage, at least 21 years old. The dataset structure is the following:

    Pregnancies: Number of times pregnant
    Glucose: Plasma glucose concentration 2 hours after an oral glucose tolerance test
    BloodPressure: Diastolic blood pressure (mm Hg)
    SkinThickness: Triceps skin fold thickness (mm)
    Insulin: 2-Hour serum insulin (mu U/ml)
    BMI: Body mass index (weight in kg/(height in m)^2)
    DiabetesPedigreeFunction: Diabetes pedigree function
    Age: Age (years)
    Outcome: Class variable (0 or 1)

In [ ]:
# importing libraries
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

## Exploratory Data Analysis

Load the `diabetes.csv` dataset (a copy can also be found in the `datasets` directory):

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv')
df.head()

Check for null values:

In [ ]:
df.isnull().sum()

There seem to be no missing values. But look at the minimum of each column:

In [ ]:
df.describe()

**Explain:** Can a living person have a `Glucose`, `BloodPressure` or `BMI` of 0? Count how many zeros there are in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI` (e.g. `(df[cols] == 0).sum()`). What do these zeros really mean, and why is `isnull()` not enough to find them?

In [ ]:
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

**Your answer:**

Show a graph with the count of examples that are `negative` and that are `positive`. You can use the [`countplot`](https://seaborn.pydata.org/generated/seaborn.countplot.html) of the `Seaborn` module, with `x="Outcome"`.

**Explain:** What percentage of the patients has diabetes? What accuracy would a "model" that always answers *no diabetes* get? Keep this number in mind when you evaluate your network.

**Your answer:**

Draw a graph showing the distribution of glucose levels. You can use the [`histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html), with `hue="Outcome"` to separate the two classes.

Do the same for the Body Mass Index (BMI) and Blood Pressure.

Draw a graph with the relation between features ([`pairplot`](https://seaborn.pydata.org/generated/seaborn.pairplot.html#seaborn.pairplot)), with `hue="Outcome"`.

**Explain:** Which feature seems to separate the two classes best? Do you see the zeros of the previous question in the plots?

**Your answer:**

## Training the model

Separate the features (`X`, every column except `Outcome`) from the target (`y`, the `Outcome` column).

In [ ]:
X = ...
y = ...

Split the dataset in two datasets: (X_train,y_train), with 80% of the examples and (X_test,y_test) with the remaining 20%. You can use [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html). Use `random_state=0` and `stratify=y`, so that both sets have the same percentage of patients with diabetes.

In [ ]:
X_train, X_test, y_train, y_test = ...

In [ ]:
print("Training samples", len(X_train), "Testing samples", len(X_test))  # should be 614 and 154

The features have very different scales (e.g. `Insulin` goes up to 846 and `DiabetesPedigreeFunction` is below 2.5). Neural networks train much faster and more reliably when the inputs have similar scales, so we **standardise** them with a [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

The scaler must be fitted (`fit_transform`) on the **training set only**, and then applied (`transform`) to the test set.

In [ ]:
scaler = StandardScaler()
X_train_s = ...
X_test_s = ...

To be able to use PyTorch, it is necessary to create [`tensors`](https://pytorch.org/docs/stable/tensors.html). The inputs are `float32`; the labels are class indices, so they must be integers (`long`) for the `CrossEntropyLoss`.

In [ ]:
# Creating Tensors
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)
y_test_t = torch.tensor(y_test.values, dtype=torch.long)

### ANN class

Create the model using two linear hidden layers ([`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)) and one linear output layer. Use the [`ReLU`](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html) activation function in the hidden layers.

* The input layer must have 8 inputs (one per feature).
* Use 20 neurons in the first hidden layer and 10 in the second.
* The output layer has 2 neurons, one per class. It must **not** have an activation function: `CrossEntropyLoss` applies the softmax itself.

In [ ]:
# Creating the Model
class ANN_model(nn.Module):
    def __init__(self, input_features=8, hidden1=20, hidden2=10, out_features=2):
        super().__init__()
        # -- Complete the code

    def forward(self, x):
        # -- Complete the code
        return x

Instantiate the class.

In [ ]:
torch.manual_seed(20)
model = ANN_model()

Confirm the architecture by printing the model and the number of parameters:

In [ ]:
print(model)
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

**Explain:** Show how the number of parameters is calculated (do not forget the biases).

**Your answer:**

Use [`CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) as the loss function and the [`Adam`](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) optimizer with $lr=0.01$.

In [ ]:
# Backward Propagation - loss and optimizer
loss_function = ...
optimizer = ...

Train for 500 epochs. In every epoch the whole training set is used at once (*full batch*). We also compute the loss on the test set, **without** computing gradients (`torch.no_grad()`), to see how the model behaves with data it has not seen.

Note that we store `loss.item()` (a Python number) and not `loss` (a tensor that keeps a reference to the whole computation graph).

In [ ]:
epochs = 500
train_losses = []
test_losses = []
for i in range(1, epochs + 1):
    model.train()
    y_pred = model(X_train_t)
    loss = loss_function(y_pred, y_train_t)
    train_losses.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_losses.append(loss_function(model(X_test_t), y_test_t).item())

    if i % 50 == 1:
        print(f"Epoch {i:3d}: train loss = {train_losses[-1]:.4f}, test loss = {test_losses[-1]:.4f}")

Plot `train_losses` and `test_losses` on the same graph, to display the cost along the training. Label the axes and add a legend.

In [ ]:
# plot the loss function

**Explain:** Describe the two curves. What happens to the test loss after some epochs, and what is this phenomenon called? How many epochs would you choose? Why should that choice **not** be made by looking at the test set (what should be used instead)?

**Your answer:**

Make the predictions on the `X_test_t` feature set and store the result in `predictions`. Use `torch.no_grad()`; the predicted class is the index of the largest output (`argmax(dim=1)`).

In [ ]:
# Calculate the predictions
predictions = ...

### Model evaluation

Display the confusion matrix using [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) and Seaborn's [`heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html). Note that in scikit-learn's confusion matrix the **rows are the true classes** and the **columns are the predicted classes**: label the axes accordingly.

In [ ]:
# Confusion matrix

Finally, display the accuracy score ([`accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)) for the training set and for the test set.

In [ ]:
# Accuracy

**Explain:** Compare the test accuracy with the baseline you computed in the exploration, and with the training accuracy. Looking at the confusion matrix, which error is more frequent: a diabetic patient classified as healthy, or the opposite? Which of the two errors is worse in this application?

**Your answer:**

### Things to try

1. Run the training again **without** the `StandardScaler` (use `X_train.values` and `X_test.values`). What happens to the speed of convergence?
2. Train for 50 epochs instead of 500. What happens to the test accuracy?